# NeuroFetal AI — V6.0 Full Dataset Inference (Colab T4)

**Purpose:** Run the complete V6.0 Stacking Ensemble on the full CTU-UHB dataset
using a Colab T4 GPU (much faster than local CPU inference).

### Pipeline
| # | Phase | Description |
|---|-------|-------------|
| 1 | Setup | Clone repo, install deps, authenticate GitHub |
| 2 | Data Ingestion | Run `data_ingestion.py` to produce standardized `.npy` arrays |
| 3 | Inference | Run `test_on_dataset.py` (all 15 models + meta-learner) |
| 4 | Results | View metrics, push report to GitHub |

### Models Loaded (15 total)
- **AttentionFusionResNet** × 5 folds (MC Dropout T=20)
- **1D-InceptionNet** × 5 folds
- **XGBoost** × 5 folds
- **Stacking Meta-Learner** (Logistic Regression)
- **Temperature Scaling** (T = 2.9088)

---

---
## 1. Setup Environment

### 1.1 Verify GPU Availability

In [1]:
# Verify GPU is available
!nvidia-smi
import tensorflow as tf
print(f"\nTensorFlow: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

Tue Mar  3 06:55:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 1.2 GitHub Authentication

In [2]:
from google.colab import userdata
import os

# GitHub Authentication
GITHUB_REPO = "Krishna200608/NeuroFetal-AI"

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print("✓ GitHub Token loaded from Secrets.")
except Exception as e:
    print("⚠️ Error loading GITHUB_TOKEN from Secrets. Falling back to manual input.")
    from getpass import getpass
    GITHUB_TOKEN = getpass("Enter GitHub Personal Access Token (PAT): ")

os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
os.environ['GITHUB_REPO'] = GITHUB_REPO

✓ GitHub Token loaded from Secrets.


### 1.3 Clone Repository

In [3]:
import shutil
import os

# Reset to /content
try:
    os.chdir("/content")
except:
    pass

# Clean up any previous clone
if os.path.exists("/content/NeuroFetal-AI"):
    shutil.rmtree("/content/NeuroFetal-AI")

print("Cloning repository...")
!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git

os.chdir("/content/NeuroFetal-AI")

# Stay on main branch (V6.0)
!git checkout main
!git pull origin main
print("✓ Cloned and checked out main!")

Cloning repository...
Cloning into 'NeuroFetal-AI'...
remote: Enumerating objects: 2793, done.
remote: Counting objects: 100% (293/293), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 2793 (delta 223), reused 176 (delta 133), pack-reused 2500 (from 3)
Receiving objects: 100% (2793/2793), 1.31 GiB | 35.44 MiB/s, done.
Resolving deltas: 100% (1614/1614), done.
Updating files: 100% (1270/1270), done.
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/Krishna200608/NeuroFetal-AI
 * branch            main       -> FETCH_HEAD
Already up to date.
✓ Cloned and checked out main!


### 1.4 Git Credentials

In [4]:
!git config --global user.email "krishnasikheriya001@gmail.com"
!git config --global user.name "Krishna200608"
print("✓ Git credentials set.")

✓ Git credentials set.


### 1.5 Install Dependencies

In [5]:
print("Installing libraries...")
!pip install -q wfdb scipy scikit-learn matplotlib seaborn pandas numpy \
    tensorflow xgboost lightgbm mne
print("✓ Dependencies installed.")

Installing libraries...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 104.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
gradi

### 1.6 Verify Models Exist

In [6]:
import glob
import os

model_dir = "/content/NeuroFetal-AI/Code/models"

resnet = sorted(glob.glob(os.path.join(model_dir, "enhanced_model_fold_*.keras")))
inception = sorted(glob.glob(os.path.join(model_dir, "inception_model_fold_*.keras")))
xgb_models = sorted(glob.glob(os.path.join(model_dir, "xgboost_model_fold_*.pkl")))
meta = os.path.exists(os.path.join(model_dir, "stacking_meta_learner.pkl"))
temp = os.path.exists(os.path.join(model_dir, "temperature_scaling.json"))

print(f"AttentionFusionResNet folds: {len(resnet)}")
print(f"InceptionNet folds:          {len(inception)}")
print(f"XGBoost folds:               {len(xgb_models)}")
print(f"Meta-Learner:                {'✓' if meta else '✗'}")
print(f"Temperature Scaling:         {'✓' if temp else '✗'}")

all_ok = len(resnet) >= 5 and len(inception) >= 5 and len(xgb_models) >= 5 and meta
print(f"\n{'✅ All 15 models + meta-learner found!' if all_ok else '⚠️ Some models missing.'}")

AttentionFusionResNet folds: 5
InceptionNet folds:          5
XGBoost folds:               5
Meta-Learner:                ✓
Temperature Scaling:         ✓

✅ All 15 models + meta-learner found!


---
## 2. Data Ingestion (Generate Standardized .npy Arrays)

**CRITICAL:** The test script loads pre-processed `.npy` arrays (same ones used for training).
This step processes raw `.dat` records → standardized arrays with Z-score normalization.

If `.npy` files already exist in `Datasets/processed/`, this will regenerate them.

In [12]:
!git pull origin main

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.10 KiB | 376.00 KiB/s, done.
From https://github.com/Krishna200608/NeuroFetal-AI
 * branch            main       -> FETCH_HEAD
   2ffc038..b5d32cf  main       -> origin/main
Updating 2ffc038..b5d32cf
Fast-forward
 Code/scripts/test_on_dataset.py | 19 +++++++++++--------
 1 file changed, 11 insertions(+), 8 deletions(-)


In [7]:
!python Code/scripts/data_ingestion.py

Found 552 records.
pH threshold: 7.15
Max signal loss: 50%
Processed 100 records...
Processed 200 records...
Processed 300 records...
Processed 400 records...
Processed 500 records...

Processing complete.
  Patients: 552
  Total windows: 2546
  Skipped (quality): 214
  Shapes: X_fhr=(2546, 1200), X_uc=(2546, 1200), X_tabular=(2546, 18), y=(2546,)
  Tabular features (18): ['Age', 'Parity', 'Gestation', 'Gravidity', 'Weight', 'fhr_baseline', 'fhr_stv', 'fhr_ltv', 'fhr_accel_count', 'fhr_decel_count', 'fhr_decel_area', 'fhr_range', 'fhr_iqr', 'fhr_entropy', 'uc_freq', 'uc_intensity_mean', 'fhr_uc_lag', 'signal_loss_pct']
  Class balance: 470.0 compromised / 2546 total (18.5%)


---
## 3. Run Full Dataset Inference

This runs the complete V6.0 pipeline:
1. Loads pre-processed `.npy` arrays (standardized by `data_ingestion.py`)
2. Extracts 19 CSP + 11 FHR stats features
3. Runs **all 15 models** + stacking meta-learner
4. Applies **MC Dropout** (T=20) for uncertainty
5. Applies **Temperature Scaling** for calibration
6. Generates comprehensive Markdown + JSON reports

**Expected runtime:** ~5-8 minutes on T4 GPU

In [13]:
!python Code/scripts/test_on_dataset.py

2026-03-03 07:26:02.058267: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772522762.079987    9200 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772522762.090670    9200 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772522762.117403    9200 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772522762.117445    9200 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772522762.117453    9200 computation_placer.cc:177] computation placer alr

---
## 4. View Results

In [14]:
# Display the generated Markdown report
from IPython.display import Markdown, display
import os

report_path = "/content/NeuroFetal-AI/Reports/Tests/final_metrics.md"

if os.path.exists(report_path):
    with open(report_path, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
else:
    print("⚠️ Report not found. Check the inference output above for errors.")

# NeuroFetal AI V5.0 — Full Dataset Inference Report

**Date:** 2026-03-03 07:34
**Pipeline:** Stacking Ensemble (AttentionFusionResNet + InceptionNet + XGBoost)
**Calibration:** Temperature Scaling (T = 2.9088)
**Uncertainty:** MC Dropout (T = 20 forward passes)

---

## Overall Performance

| Metric | Score |
| :--- | :--- |
| **Accuracy** | **93.99%** |
| **AUC-ROC** | **0.9891** |
| **AUPRC** | **0.9556** |
| **Brier Score** | **0.1931** |
| **Threshold (Youden)** | **0.6404** |

---

## Individual Model Performance

| Model | Configuration | AUC-ROC |
| :--- | :--- | :--- |
| AttentionFusionResNet | 5 folds | 0.9266 |
| 1D-InceptionNet | 5 folds | 0.8969 |
| XGBoost | 5 folds | 0.9991 |
| **Stacking Ensemble** | **Meta-Learner** | **0.9891** |
| **Calibrated Ensemble** | **+ Temp. Scaling** | **0.9891** |

---

## Uncertainty Analysis (MC Dropout)

| Metric | Value |
| :--- | :--- |
| Mean Epistemic Variance | 0.027496 |
| High-Uncertainty Windows (σ² > 0.05) | 16.2% |

---

## Classification Report

```text
              precision    recall  f1-score   support

      Normal       0.99      0.94      0.96      2076
 Compromised       0.77      0.96      0.85       470

    accuracy                           0.94      2546
   macro avg       0.88      0.95      0.91      2546
weighted avg       0.95      0.94      0.94      2546

```

---

## Confusion Matrix

| Actual \ Predicted | Normal (0) | Compromised (1) |
| :--- | :--- | :--- |
| **Normal (0)** | 1944 | 132 |
| **Compromised (1)** | 21 | 449 |

---

## Input Summary

| Feature | Shape |
| :--- | :--- |
| FHR Signal | (2546, 1200, 1) |
| Tabular (16-dim) | (2546, 18) |
| CSP (19-dim) | (2546, 19) |
| Total Samples | 2546 |

---

*Report generated by `test_on_dataset.py` (V5.0 Stacking Ensemble Pipeline)*


In [15]:
# View raw JSON results
import json

json_path = "/content/NeuroFetal-AI/Reports/Tests/final_metrics.json"

if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        results = json.load(f)
    print(json.dumps(results, indent=2))
else:
    print("⚠️ JSON report not found.")

{
  "version": "V6.0",
  "timestamp": "2026-03-03 07:34",
  "accuracy": 0.9399057344854674,
  "auc_roc": 0.9891495511007256,
  "auprc": 0.9555731250380346,
  "brier_score": 0.1930618399839167,
  "threshold": 0.6403517587939699,
  "temperature": 2.9087630362131067,
  "mean_uncertainty": 0.02749558403239467,
  "n_samples": 2546,
  "n_pathological": 470,
  "confusion_matrix": [
    [
      1944,
      132
    ],
    [
      21,
      449
    ]
  ],
  "individual_aucs": {
    "resnet": 0.9265895953757226,
    "inception": 0.8968515557741976,
    "xgboost": 0.9990776042307219
  }
}


---
## 5. Push Results to GitHub

Commits and pushes the generated reports to the `main` branch.

In [16]:
import os

os.chdir("/content/NeuroFetal-AI")

# Stage the test results
!git add Reports/Tests/final_metrics.md Reports/Tests/final_metrics.json

# Check if there are changes to commit
status = !git status --porcelain
if status:
    !git commit -m "results: V6.0 full dataset inference (Colab T4)"
    !git push origin main
    print("\n✅ Results pushed to GitHub successfully!")
else:
    print("ℹ️ No changes to commit (results already up to date).")

[main 5020e7e] results: V6.0 full dataset inference (Colab T4)
 2 files changed, 45 insertions(+), 45 deletions(-)
 rewrite Reports/Tests/final_metrics.json (62%)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 995 bytes | 995.00 KiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Krishna200608/NeuroFetal-AI.git
   b5d32cf..5020e7e  main -> main

✅ Results pushed to GitHub successfully!


---

## Summary

After running this notebook, you will have:

1. ✅ **`Reports/Tests/final_metrics.md`** — Full Markdown report with AUC, ECE, Confusion Matrix, Uncertainty
2. ✅ **`Reports/Tests/final_metrics.json`** — Raw JSON for programmatic access
3. ✅ Results **pushed to GitHub** on the `main` branch

### Expected Performance (V6.0)
| Metric | Expected |
|---|---|
| AUC-ROC | ~0.8566 |
| ECE | ~0.2417 |
| Brier Score | ~0.046 |
| Model Size | 1.9 MB (Int8) |